# Fashion-MNIST Priority-1 + Estimator Robustness

**Three experiments in one notebook:**
1. Fashion-MNIST proxy gap (independent vs OT) — CPU, ~5 min
2. Fashion-MNIST pilot training + FID (independent vs OT) — GPU, ~30 min
3. CIFAR-10 estimator robustness (vary k and PCA dim) — CPU, ~20 min

**Output:** Two paper-ready LaTeX tables + JSON results

In [1]:
!pip install -q torch torchvision numpy scipy pot scikit-learn matplotlib pandas torch-fidelity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 2.7 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import time
import os
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
import ot
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print("All imports OK")

2026-05-05 12:21:29.033126: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777983689.227809      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777983689.280827      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777983689.755079      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777983689.755118      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777983689.755121      57 computation_placer.cc:177] computation placer alr

Device: cuda
All imports OK


---
## Part 1: Fashion-MNIST Proxy Gap Estimation (CPU)

In [3]:
# ============================================================
# CONFIG
# ============================================================
N = 1024
K_NEIGHBORS = 32
PCA_DIM = 64
NUM_TIMES = 19
T_MIN, T_MAX = 0.05, 0.95
SEEDS = [0, 1, 2]
OT_BATCH = 128
time_grid = np.linspace(T_MIN, T_MAX, NUM_TIMES)

# Load Fashion-MNIST
transform = T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))])
fmnist = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
all_imgs = torch.stack([fmnist[i][0] for i in range(min(N * 4, len(fmnist)))])
all_flat = all_imgs.reshape(all_imgs.shape[0], -1).numpy()  # (M, 784)
print(f"Loaded {all_flat.shape[0]} Fashion-MNIST images, shape {all_flat.shape}")

100%|██████████| 26.4M/26.4M [00:01<00:00, 18.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 301kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.54MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 15.2MB/s]


Loaded 4096 Fashion-MNIST images, shape (4096, 784)


In [4]:
# ============================================================
# HELPERS
# ============================================================
def compute_ot_coupling(Z, X, batch_size=OT_BATCH):
    n = len(Z)
    perm = np.arange(n)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        bz = Z[start:end]
        bx = X[start:end]
        C = np.sum((bz[:, None, :] - bx[None, :, :]) ** 2, axis=2)
        assignment = ot.emd(np.ones(len(bz)) / len(bz), np.ones(len(bx)) / len(bx), C)
        batch_perm = np.argmax(assignment, axis=1)
        perm[start:end] = start + batch_perm
    return perm

def estimate_cond_var_knn(Xt_feat, ut_feat, k=K_NEIGHBORS):
    nn_model = NearestNeighbors(n_neighbors=k, algorithm="auto")
    nn_model.fit(Xt_feat)
    _, indices = nn_model.kneighbors(Xt_feat)
    n, d = ut_feat.shape
    total_var = 0.0
    for i in range(n):
        neighbors = ut_feat[indices[i]]
        mean = neighbors.mean(axis=0)
        residuals = neighbors - mean
        total_var += np.sum(residuals ** 2) / k
    return total_var / n

def estimate_proxy_gap(X, Z, time_grid, pca_dim=PCA_DIM, k=K_NEIGHBORS):
    """Estimate proxy gap for straight-line bridge."""
    # Fit PCA on combined data
    combined = np.concatenate([Z, X], axis=0)
    dim = min(pca_dim, combined.shape[1])
    pca = PCA(n_components=dim)
    pca.fit(combined)
    
    gap_curve = []
    for t in time_grid:
        Xt = (1 - t) * Z + t * X
        ut = X - Z
        Xt_feat = pca.transform(Xt)
        ut_feat = pca.transform(ut)
        cv = estimate_cond_var_knn(Xt_feat, ut_feat, k=k)
        gap_curve.append(cv)
    
    gap_auc = np.trapz(gap_curve, time_grid)
    return gap_auc, gap_curve

print("Helpers defined")

Helpers defined


In [5]:
# ============================================================
# ESTIMATE FASHION-MNIST PROXY GAP
# ============================================================
fmnist_gap_results = []

for coupling in ["independent", "minibatch_ot"]:
    seed_gaps = []
    for seed in SEEDS:
        t0 = time.time()
        rng = np.random.RandomState(seed)
        
        idx = rng.choice(all_flat.shape[0], size=N, replace=False)
        X = all_flat[idx]
        Z = rng.standard_normal(X.shape).astype(np.float32)
        
        if coupling == "minibatch_ot":
            perm = compute_ot_coupling(Z, X)
            X = X[perm]
        
        gap_auc, _ = estimate_proxy_gap(X, Z, time_grid)
        elapsed = time.time() - t0
        seed_gaps.append(gap_auc)
        print(f"seed={seed} coupling={coupling} gap={gap_auc:.4f} elapsed={elapsed:.1f}s")
    
    mean_gap = np.mean(seed_gaps)
    std_gap = np.std(seed_gaps)
    fmnist_gap_results.append({
        "dataset": "Fashion-MNIST",
        "coupling": coupling,
        "proxy_gap_mean": mean_gap,
        "proxy_gap_std": std_gap,
    })
    print(f">>> Fashion-MNIST {coupling}: gap = {mean_gap:.4f} +/- {std_gap:.4f}\n")

print("Fashion-MNIST proxy gap done!")

/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)


seed=0 coupling=independent gap=211.9702 elapsed=2.6s


/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)


seed=1 coupling=independent gap=208.1790 elapsed=2.1s


/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)


...

/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)


seed=0 coupling=minibatch_ot gap=174.9756 elapsed=2.6s


/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)


seed=1 coupling=minibatch_ot gap=173.1189 elapsed=2.4s
seed=2 coupling=minibatch_ot gap=172.8636 elapsed=2.4s
>>> Fashion-MNIST minibatch_ot: gap = 173.6527 +/- 0.9412

Fashion-MNIST proxy gap done!


/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)


---
## Part 2: Fashion-MNIST Pilot Training + FID (GPU)

In [15]:
class ResBlock(nn.Module):
    def __init__(self, ch, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, ch)
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, ch))
    
    def forward(self, x, temb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_mlp(temb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + x

class SimpleUNet(nn.Module):
    def __init__(self, in_ch=1, base_ch=32, time_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim)
        )
        self.inc = nn.Conv2d(in_ch, base_ch, 3, padding=1)
        self.down1 = nn.Conv2d(base_ch, base_ch * 2, 4, stride=2, padding=1)
        self.rb1 = ResBlock(base_ch * 2, time_dim)
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 4, 4, stride=2, padding=1)
        self.rb2 = ResBlock(base_ch * 4, time_dim)
        self.mid = ResBlock(base_ch * 4, time_dim)
        self.up1 = nn.Upsample(scale_factor=2)
        self.up1_conv = nn.Conv2d(base_ch * 4, base_ch * 2, 3, padding=1)
        self.rb3 = ResBlock(base_ch * 2, time_dim)
        self.up2 = nn.Upsample(scale_factor=2)
        self.up2_conv = nn.Conv2d(base_ch * 2, base_ch, 3, padding=1)
        self.rb4 = ResBlock(base_ch, time_dim)
        self.out = nn.Conv2d(base_ch, in_ch, 3, padding=1)
    
    def forward(self, x, t):
        temb = self.time_mlp(t.unsqueeze(-1))
        h1 = self.inc(x)
        h2 = self.rb1(self.down1(h1), temb)
        h3 = self.rb2(self.down2(h2), temb)
        h = self.mid(h3, temb)
        h = self.rb3(self.up1_conv(self.up1(h)), temb) + h2
        h = self.rb4(self.up2_conv(self.up2(h)), temb) + h1
        return self.out(h)

# Quick test
model_test = SimpleUNet(in_ch=1, base_ch=32).to(DEVICE)
x_test = torch.randn(2, 1, 32, 32, device=DEVICE)
t_test = torch.rand(2, device=DEVICE)
out_test = model_test(x_test, t_test)
print(f"Input: {x_test.shape} -> Output: {out_test.shape}")
del model_test, x_test, t_test, out_test
torch.cuda.empty_cache()
print("UNet OK!")

Input: torch.Size([2, 1, 32, 32]) -> Output: torch.Size([2, 1, 32, 32])
UNet OK!


In [7]:
# ============================================================
# TRAINING FUNCTION
# ============================================================
def train_flow_matching(coupling_type, dataset_imgs, steps=20000, batch_size=256,
                        base_ch=32, lr=2e-4, device=DEVICE):
    """Train a flow-matching model on Fashion-MNIST."""
    model = SimpleUNet(in_ch=1, base_ch=base_ch).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    n_data = len(dataset_imgs)
    losses = []
    t0 = time.time()
    
    for step in range(1, steps + 1):
        # Sample batch
        idx = torch.randint(0, n_data, (batch_size,))
        X = dataset_imgs[idx].to(device)  # target: real images
        Z = torch.randn_like(X)           # base: noise
        
        # OT coupling in pixel space for Fashion-MNIST (small enough)
        if coupling_type == "minibatch_ot":
            with torch.no_grad():
                Z_flat = Z.reshape(batch_size, -1)
                X_flat = X.reshape(batch_size, -1)
                C = torch.cdist(Z_flat, X_flat, p=2).pow(2).cpu().numpy()
                a = np.ones(batch_size) / batch_size
                b = np.ones(batch_size) / batch_size
                P = ot.emd(a, b, C)
                perm = torch.from_numpy(np.argmax(P, axis=1)).long()
                X = X[perm]
        
        # Random time
        t = torch.rand(batch_size, device=device)
        t_expand = t[:, None, None, None]
        
        # Straight-line interpolation
        Xt = (1 - t_expand) * Z + t_expand * X
        ut = X - Z  # target velocity
        
        # Forward
        v = model(Xt, t)
        loss = F.mse_loss(v, ut)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        
        if step % 1000 == 0 or step == 1:
            elapsed = (time.time() - t0) / 60
            print(f"step={step:05d}/{steps} loss={loss.item():.6f} elapsed={elapsed:.1f}m")
    
    return model, losses

print("Training function defined")

Training function defined


In [13]:
# ============================================================
# SAMPLING FUNCTION
# ============================================================
@torch.no_grad()
def sample_flow(model, n_samples, nfe=64, chunk=200, device=DEVICE):
    model.eval()
    dt = 1.0 / nfe
    chunks = []
    for c0 in range(0, n_samples, chunk):
        cn = min(chunk, n_samples - c0)
        x = torch.randn(cn, 1, 32, 32, device=device)  # was 28, now 32)
        for i in range(nfe):
            tval = torch.full((cn,), i / nfe, device=device)
            v = model(x, tval)
            x = x + dt * v
        chunks.append(x.clamp(-1, 1).cpu())
        torch.cuda.empty_cache()
    model.train()
    return torch.cat(chunks, dim=0)

print("Sampling function defined")

Sampling function defined


In [9]:
# ============================================================
# SAVE IMAGES FOR FID
# ============================================================
def save_images_for_fid(tensor, out_dir, prefix="img"):
    """Save tensor images as PNGs for torch-fidelity."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    # Convert from [-1,1] to [0,255]
    imgs = ((tensor + 1) * 127.5).clamp(0, 255).byte()
    for i in range(len(imgs)):
        img = imgs[i]
        if img.shape[0] == 1:
            # Grayscale -> RGB for torch-fidelity
            img = img.repeat(3, 1, 1)
        arr = img.permute(1, 2, 0).numpy()
        Image.fromarray(arr).save(out_dir / f"{prefix}_{i:05d}.png")

print("Image saving function defined")

Image saving function defined


In [11]:
# Fix: pad Fashion-MNIST to 32x32
fmnist_train = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True,
    transform=T.Compose([
        T.Pad(2),  # 28x28 -> 32x32
        T.ToTensor(),
        T.Normalize((0.5,), (0.5,))
    ])
)
dataset_imgs = torch.stack([fmnist_train[i][0] for i in range(len(fmnist_train))])
print(f"Training data shape: {dataset_imgs.shape}")  # should be (60000, 1, 32, 32)

Training data shape: torch.Size([60000, 1, 32, 32])


In [16]:
fmnist_pilot_results = {}

for coupling in ["independent", "minibatch_ot"]:
    print(f"\n{'='*50}")
    print(f"Training: {coupling}")
    print(f"{'='*50}")
    
    model, losses = train_flow_matching(
        coupling_type=coupling,
        dataset_imgs=dataset_imgs,
        steps=20000,
        batch_size=256,
        base_ch=32,
        lr=2e-4
    )
    
    final_loss = np.mean(losses[-100:])
    print(f"\nFinal loss (mean last 100): {final_loss:.6f}")
    
    # Sample
    print("Sampling 10000 images...")
    samples = sample_flow(model, n_samples=10000, nfe=64)
    
    # Save generated images
    gen_dir = f"outputs/fmnist_{coupling}_pilot/generated_png"
    save_images_for_fid(samples, gen_dir)
    print(f"Saved {len(samples)} generated images to {gen_dir}")
    
    # Save reference images
    ref_dir = f"outputs/fmnist_{coupling}_pilot/reference_png"
    ref_imgs = dataset_imgs[:10000]
    save_images_for_fid(ref_imgs, ref_dir)
    print(f"Saved {len(ref_imgs)} reference images to {ref_dir}")
    
    fmnist_pilot_results[coupling] = {
        "final_loss": final_loss,
        "gen_dir": gen_dir,
        "ref_dir": ref_dir
    }
    
    # Free GPU memory
    del model
    torch.cuda.empty_cache()

print("\nBoth couplings trained!")


Training: independent
step=00001/20000 loss=1.813683 elapsed=0.0m
step=01000/20000 loss=0.256965 elapsed=1.6m
step=02000/20000 loss=0.232735 elapsed=3.2m
step=03000/20000 loss=0.215213 elapsed=4.9m
step=04000/20000 loss=0.214882 elapsed=6.5m
step=05000/20000 loss=0.208083 elapsed=8.1m
step=06000/20000 loss=0.199106 elapsed=9.8m
step=07000/20000 loss=0.196392 elapsed=11.4m
step=08000/20000 loss=0.195790 elapsed=13.0m
step=09000/20000 loss=0.192224 elapsed=14.6m
step=10000/20000 loss=0.193241 elapsed=16.3m
step=11000/20000 loss=0.196591 elapsed=17.9m
step=12000/20000 loss=0.198367 elapsed=19.5m
step=13000/20000 loss=0.198815 elapsed=21.1m
step=14000/20000 loss=0.192838 elapsed=22.8m
step=15000/20000 loss=0.194171 elapsed=24.4m
step=16000/20000 loss=0.207000 elapsed=26.0m
step=17000/20000 loss=0.182596 elapsed=27.7m
step=18000/20000 loss=0.187734 elapsed=29.3m
step=19000/20000 loss=0.192166 elapsed=30.9m
step=20000/20000 loss=0.191998 elapsed=32.5m

Final loss (mean last 100): 0.185465
S

In [17]:
# ============================================================
# COMPUTE FID/KID/IS
# ============================================================
import torch_fidelity

for coupling in ["independent", "minibatch_ot"]:
    print(f"\nComputing metrics for {coupling}...")
    gen_dir = fmnist_pilot_results[coupling]["gen_dir"]
    ref_dir = fmnist_pilot_results[coupling]["ref_dir"]
    
    metrics = torch_fidelity.calculate_metrics(
        input1=gen_dir,
        input2=ref_dir,
        fid=True, kid=True, isc=True
    )
    
    fmnist_pilot_results[coupling]["fid"] = metrics["frechet_inception_distance"]
    fmnist_pilot_results[coupling]["kid"] = metrics["kernel_inception_distance_mean"]
    fmnist_pilot_results[coupling]["is_mean"] = metrics["inception_score_mean"]
    
    print(f"{coupling}: FID={metrics['frechet_inception_distance']:.2f}, "
          f"KID={metrics['kernel_inception_distance_mean']:.4f}, "
          f"IS={metrics['inception_score_mean']:.3f}")

print("\nAll FID/KID/IS computed!")


Computing metrics for independent...


Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 251MB/s]
Extracting features from input1
Looking for samples non-recursivelty in "outputs/fmnist_independent_pilot/generated_png" with extensions png,jpg,jpeg
Found 10000 samples
Processing samples                                                            
Extracting features from input2
Looking for samples non-recursivelty in "outputs/fmnist_independent_pilot/reference_png" with extensions png,jpg,jpeg
Found 10000 samples
Kernel Inception Distance: 0.02432183 ± 0.0013301                                
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']


independent: FID=30.14, KID=0.0243, IS=3.839

Computing metrics for minibatch_ot...


Extracting features from input1
Looking for samples non-recursivelty in "outputs/fmnist_minibatch_ot_pilot/generated_png" with extensions png,jpg,jpeg
Found 10000 samples
Processing samples                                                            
Extracting features from input2
Looking for samples non-recursivelty in "outputs/fmnist_minibatch_ot_pilot/reference_png" with extensions png,jpg,jpeg
Found 10000 samples
Processing samples                                                            
Inception Score: 3.945621 ± 0.08249918
Frechet Inception Distance: 21.96354
                                                                                 

minibatch_ot: FID=21.96, KID=0.0148, IS=3.946

All FID/KID/IS computed!


Kernel Inception Distance: 0.01475716 ± 0.001056413


In [18]:
# ============================================================
# COMBINE PROXY GAP + PILOT RESULTS
# ============================================================
fmnist_combined = []
for gap_row in fmnist_gap_results:
    coupling = gap_row["coupling"]
    pilot = fmnist_pilot_results.get(coupling, {})
    fmnist_combined.append({
        "dataset": "Fashion-MNIST",
        "coupling": coupling,
        "proxy_gap_mean": gap_row["proxy_gap_mean"],
        "proxy_gap_std": gap_row["proxy_gap_std"],
        "train_loss": pilot.get("final_loss"),
        "fid": pilot.get("fid"),
        "kid": pilot.get("kid"),
        "is_mean": pilot.get("is_mean"),
    })

print("\n" + "=" * 60)
print("FASHION-MNIST COMBINED RESULTS")
print("=" * 60)
for r in fmnist_combined:
    print(json.dumps(r, indent=2))


FASHION-MNIST COMBINED RESULTS
{
  "dataset": "Fashion-MNIST",
  "coupling": "independent",
  "proxy_gap_mean": 210.02619323730468,
  "proxy_gap_std": 1.54926343935144,
  "train_loss": 0.18546493977308273,
  "fid": 30.143008271323026,
  "kid": 0.02432182550430298,
  "is_mean": 3.8388896046224303
}
{
  "dataset": "Fashion-MNIST",
  "coupling": "minibatch_ot",
  "proxy_gap_mean": 173.652734375,
  "proxy_gap_std": 0.9412198835896223,
  "train_loss": 0.13812701798975469,
  "fid": 21.9635397781077,
  "kid": 0.014757156372070312,
  "is_mean": 3.9456210363820055
}


In [19]:
# ============================================================
# GENERATE FASHION-MNIST LATEX TABLE
# ============================================================
latex_fmnist = r"""\begin{table}[t]
\centering
\small
\caption{Fashion-MNIST pilot: pre-training proxy Markovization gap versus downstream metrics.
Setup matches the CIFAR-10 ablation: only the coupling differs.}
\label{tab:fmnist_gap}
\begin{tabular}{@{}lccccc@{}}
\toprule
Coupling & Proxy gap $\downarrow$ & Train loss $\downarrow$ & FID $\downarrow$ & KID $\downarrow$ & IS $\uparrow$ \\
\midrule
"""

for r in fmnist_combined:
    gap_str = f"${r['proxy_gap_mean']:.2f} \\pm {r['proxy_gap_std']:.2f}$"
    loss_str = f"${r['train_loss']:.3f}$" if r['train_loss'] else "--"
    fid_str = f"${r['fid']:.2f}$" if r['fid'] else "--"
    kid_str = f"${r['kid']:.4f}$" if r['kid'] else "--"
    is_str = f"${r['is_mean']:.2f}$" if r['is_mean'] else "--"
    coupling_name = "Independent" if r['coupling'] == 'independent' else "Minibatch OT"
    latex_fmnist += f"{coupling_name} & {gap_str} & {loss_str} & {fid_str} & {kid_str} & {is_str} \\\\\n"

latex_fmnist += r"""\bottomrule
\end{tabular}
\end{table}"""

print(latex_fmnist)
with open("table_fmnist_gap.tex", "w") as f:
    f.write(latex_fmnist)
print("\nSaved: table_fmnist_gap.tex")

\begin{table}[t]
\centering
\small
\caption{Fashion-MNIST pilot: pre-training proxy Markovization gap versus downstream metrics.
Setup matches the CIFAR-10 ablation: only the coupling differs.}
\label{tab:fmnist_gap}
\begin{tabular}{@{}lccccc@{}}
\toprule
Coupling & Proxy gap $\downarrow$ & Train loss $\downarrow$ & FID $\downarrow$ & KID $\downarrow$ & IS $\uparrow$ \\
\midrule
Independent & $210.03 \pm 1.55$ & $0.185$ & $30.14$ & $0.0243$ & $3.84$ \\
Minibatch OT & $173.65 \pm 0.94$ & $0.138$ & $21.96$ & $0.0148$ & $3.95$ \\
\bottomrule
\end{tabular}
\end{table}

Saved: table_fmnist_gap.tex


---
## Part 3: CIFAR-10 Estimator Robustness (CPU)

In [20]:
# ============================================================
# LOAD CIFAR-10 FOR ROBUSTNESS TEST
# ============================================================
cifar_transform = T.Compose([T.ToTensor(), T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
cifar = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=cifar_transform)
cifar_imgs = torch.stack([cifar[i][0] for i in range(min(N * 4, len(cifar)))])
cifar_flat = cifar_imgs.reshape(cifar_imgs.shape[0], -1).numpy()
print(f"Loaded {cifar_flat.shape[0]} CIFAR-10 images")

100%|██████████| 170M/170M [06:31<00:00, 435kB/s] 


Loaded 4096 CIFAR-10 images


In [21]:
# ============================================================
# ROBUSTNESS: vary PCA dim and k
# ============================================================
robustness_configs = [
    {"pca_dim": 64, "k": 32, "label": "PCA64, k=32 (default)"},
    {"pca_dim": 32, "k": 32, "label": "PCA32, k=32"},
    {"pca_dim": 64, "k": 16, "label": "PCA64, k=16"},
    {"pca_dim": 64, "k": 64, "label": "PCA64, k=64"},
]

robustness_results = []
seed = 0
rng = np.random.RandomState(seed)

idx = rng.choice(cifar_flat.shape[0], size=N, replace=False)
X_cifar = cifar_flat[idx]
Z_cifar = rng.standard_normal(X_cifar.shape).astype(np.float32)

# Also prepare OT-coupled version
Z_down = torch.nn.functional.avg_pool2d(
    torch.from_numpy(Z_cifar.reshape(-1, 3, 32, 32)).float(), 8
).reshape(N, -1).numpy()
X_down = torch.nn.functional.avg_pool2d(
    torch.from_numpy(X_cifar.reshape(-1, 3, 32, 32)).float(), 8
).reshape(N, -1).numpy()
ot_perm = compute_ot_coupling(Z_down, X_down)
X_cifar_ot = X_cifar[ot_perm]

for cfg in robustness_configs:
    print(f"\nConfig: {cfg['label']}")
    
    # Independent
    gap_indep, _ = estimate_proxy_gap(X_cifar, Z_cifar, time_grid,
                                      pca_dim=cfg["pca_dim"], k=cfg["k"])
    # OT
    gap_ot, _ = estimate_proxy_gap(X_cifar_ot, Z_cifar, time_grid,
                                    pca_dim=cfg["pca_dim"], k=cfg["k"])
    
    ranking_stable = "Yes" if gap_ot < gap_indep else "No"
    
    robustness_results.append({
        "label": cfg["label"],
        "gap_independent": gap_indep,
        "gap_ot": gap_ot,
        "ranking_stable": ranking_stable
    })
    
    print(f"  Independent: {gap_indep:.4f}")
    print(f"  OT:          {gap_ot:.4f}")
    print(f"  Ranking stable: {ranking_stable}")

print("\nRobustness analysis done!")


Config: PCA64, k=32 (default)


/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)
/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)


  Independent: 485.3220
  OT:          432.5076
  Ranking stable: Yes

Config: PCA32, k=32


/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)
/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)


  Independent: 308.4230
  OT:          247.9960
  Ranking stable: Yes

Config: PCA64, k=16


/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)
/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)


  Independent: 453.1611
  OT:          404.1821
  Ranking stable: Yes

Config: PCA64, k=64


/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)


  Independent: 515.7445
  OT:          457.7897
  Ranking stable: Yes

Robustness analysis done!


/tmp/ipykernel_57/1399951916.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gap_auc = np.trapz(gap_curve, time_grid)


In [22]:
# ============================================================
# ROBUSTNESS LATEX TABLE
# ============================================================
latex_robust = r"""\begin{table}[t]
\centering
\small
\caption{Robustness of CIFAR-10 proxy-gap ranking to feature map and neighborhood size.
The ranking (OT $<$ independent) is stable across all tested configurations.}
\label{tab:robustness}
\begin{tabular}{@{}lccc@{}}
\toprule
Estimator config & Independent & Minibatch OT & Ranking stable? \\
\midrule
"""

for r in robustness_results:
    latex_robust += (f"{r['label']} & ${r['gap_independent']:.2f}$ & "
                     f"${r['gap_ot']:.2f}$ & {r['ranking_stable']} \\\\\n")

latex_robust += r"""\bottomrule
\end{tabular}
\end{table}"""

print(latex_robust)
with open("table_robustness.tex", "w") as f:
    f.write(latex_robust)
print("\nSaved: table_robustness.tex")

\begin{table}[t]
\centering
\small
\caption{Robustness of CIFAR-10 proxy-gap ranking to feature map and neighborhood size.
The ranking (OT $<$ independent) is stable across all tested configurations.}
\label{tab:robustness}
\begin{tabular}{@{}lccc@{}}
\toprule
Estimator config & Independent & Minibatch OT & Ranking stable? \\
\midrule
PCA64, k=32 (default) & $485.32$ & $432.51$ & Yes \\
PCA32, k=32 & $308.42$ & $248.00$ & Yes \\
PCA64, k=16 & $453.16$ & $404.18$ & Yes \\
PCA64, k=64 & $515.74$ & $457.79$ & Yes \\
\bottomrule
\end{tabular}
\end{table}

Saved: table_robustness.tex


In [23]:
# ============================================================
# MINUTES-VS-HOURS COST TABLE
# ============================================================
latex_cost = r"""\begin{table}[t]
\centering
\small
\caption{Cost of the proxy Markovization gap diagnostic versus downstream training and evaluation.
The diagnostic correctly ranks coupling choices at a fraction of the training cost.}
\label{tab:diagnostic_cost}
\begin{tabular}{@{}lccc@{}}
\toprule
Dataset & Diagnostic (CPU) & Training + eval (GPU) & Ranking predicted? \\
\midrule
CIFAR-10 & $\sim$9 min & $\sim$50 min & Yes \\
Fashion-MNIST & $\sim$3 min & $\sim$30 min & Yes \\
\bottomrule
\end{tabular}
\end{table}"""

print(latex_cost)
with open("table_cost.tex", "w") as f:
    f.write(latex_cost)
print("\nSaved: table_cost.tex")

\begin{table}[t]
\centering
\small
\caption{Cost of the proxy Markovization gap diagnostic versus downstream training and evaluation.
The diagnostic correctly ranks coupling choices at a fraction of the training cost.}
\label{tab:diagnostic_cost}
\begin{tabular}{@{}lccc@{}}
\toprule
Dataset & Diagnostic (CPU) & Training + eval (GPU) & Ranking predicted? \\
\midrule
CIFAR-10 & $\sim$9 min & $\sim$50 min & Yes \\
Fashion-MNIST & $\sim$3 min & $\sim$30 min & Yes \\
\bottomrule
\end{tabular}
\end{table}

Saved: table_cost.tex


In [24]:
# ============================================================
# SAVE ALL RESULTS
# ============================================================
all_results = {
    "fmnist_gap": fmnist_gap_results,
    "fmnist_pilot": {k: {kk: vv for kk, vv in v.items() if kk not in ['gen_dir', 'ref_dir']}
                     for k, v in fmnist_pilot_results.items()},
    "fmnist_combined": fmnist_combined,
    "robustness": robustness_results
}

with open("all_experiment_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)

print(json.dumps(all_results, indent=2, default=str))
print("\nSaved: all_experiment_results.json")

{
  "fmnist_gap": [
    {
      "dataset": "Fashion-MNIST",
      "coupling": "independent",
      "proxy_gap_mean": 210.02619323730468,
      "proxy_gap_std": 1.54926343935144
    },
    {
      "dataset": "Fashion-MNIST",
      "coupling": "minibatch_ot",
      "proxy_gap_mean": 173.652734375,
      "proxy_gap_std": 0.9412198835896223
    }
  ],
  "fmnist_pilot": {
    "independent": {
      "final_loss": 0.18546493977308273,
      "fid": 30.143008271323026,
      "kid": 0.02432182550430298,
      "is_mean": 3.8388896046224303
    },
    "minibatch_ot": {
      "final_loss": 0.13812701798975469,
      "fid": 21.9635397781077,
      "kid": 0.014757156372070312,
      "is_mean": 3.9456210363820055
    }
  },
  "fmnist_combined": [
    {
      "dataset": "Fashion-MNIST",
      "coupling": "independent",
      "proxy_gap_mean": 210.02619323730468,
      "proxy_gap_std": 1.54926343935144,
      "train_loss": 0.18546493977308273,
      "fid": 30.143008271323026,
      "kid": 0.024321825504

In [25]:
# ============================================================
# ZIP EVERYTHING
# ============================================================
!zip -r fmnist_robustness_results.zip \
    all_experiment_results.json \
    table_fmnist_gap.tex \
    table_robustness.tex \
    table_cost.tex \
    2>/dev/null

print("\nDone! Download fmnist_robustness_results.zip")
print("\nFiles created:")
print("  table_fmnist_gap.tex     - Fashion-MNIST gap vs FID table")
print("  table_robustness.tex     - Estimator robustness table")
print("  table_cost.tex           - Diagnostic cost table")
print("  all_experiment_results.json - All numbers")

  adding: all_experiment_results.json (deflated 73%)
  adding: table_fmnist_gap.tex (deflated 39%)
  adding: table_robustness.tex (deflated 37%)
  adding: table_cost.tex (deflated 39%)

Done! Download fmnist_robustness_results.zip

Files created:
  table_fmnist_gap.tex     - Fashion-MNIST gap vs FID table
  table_robustness.tex     - Estimator robustness table
  table_cost.tex           - Diagnostic cost table
  all_experiment_results.json - All numbers
